In [11]:
import zipfile
import os

# Define file and destination path
zip_path = "archive.zip"
extract_path = "unzipped_archive"

# Create folder and extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Archive extracted to:", extract_path)


✅ Archive extracted to: unzipped_archive


In [ ]:
import kagglehub
borhanitrash_football_players_detection_dataset_path = kagglehub.dataset_download('borhanitrash/football-players-detection-dataset')

print('Data source import complete.')


100%|██████████| 68.0M/68.0M [00:00<00:00, 185MB/s]

Extracting files...


Data source import complete.


In [4]:
!pip install ultralytics albumentations -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.3 MB/s eta 0:00:00


In [6]:
import os
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import glob
from IPython.display import display, Image
import numpy as np
import yaml
from PIL import Image as PILImage
import albumentations as A
import shutil
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [10]:
if torch.cuda.is_available():

    print("GPU is available!")

    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

else:

    print("GPU not available. Using CPU.")

GPU is available!
Using GPU: Tesla T4


In [12]:
main_path = '/content/archive'
data_yaml_path = '/content/archive/data.yaml'
class_names = ["ball", "goalkeeper", "player", "referee"]

In [13]:
work_dir = '/content/football_detection_improved'
os.makedirs(work_dir, exist_ok=True)

In [14]:
with open(data_yaml_path, 'r') as file:
    data_cfg = yaml.safe_load(file)

print("Dataset configuration:")
print(data_cfg)

Dataset configuration:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['ball', 'goalkeeper', 'player', 'referee'], 'roboflow': {'workspace': 'roboflow-jvuqo', 'project': 'football-players-detection-3zvbc', 'version': 9, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/9'}}


In [15]:
model_variant = 'yolov8m.pt'

In [16]:
IMG_SIZE = 1280
BATCH_SIZE = 8

EPOCHS = 50

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0005
CONF_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.5

In [17]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [22]:
custom_data_yaml = os.path.join(work_dir, 'custom_data.yaml')

with open(data_yaml_path, 'r') as infile, open(custom_data_yaml, 'w') as outfile:
    data = yaml.safe_load(infile)

    # Add class weights to emphasize rare classes more during training
    data['class_weights'] = [5.0, 3.0, 1.0, 4.0]  # Increase weights for ball(0), goalkeeper(1), and referee(3)

    yaml.dump(data, outfile)


In [23]:
def create_augmented_dataset():
    """Create a copy and augment data for rare classes (ball, goalkeeper, referee)"""

    # Create directory for the augmented dataset
    aug_dir = os.path.join(work_dir, 'augmented_dataset')

    # Create YOLOv8-compliant folder structure
    train_images_dir = os.path.join(aug_dir, 'train', 'images')
    train_labels_dir = os.path.join(aug_dir, 'train', 'labels')
    val_images_dir = os.path.join(aug_dir, 'valid', 'images')  # Folder name must be "valid", not "val"
    val_labels_dir = os.path.join(aug_dir, 'valid', 'labels')

    # Create necessary folders
    os.makedirs(train_images_dir, exist_ok=True)
    os.makedirs(train_labels_dir, exist_ok=True)
    os.makedirs(val_images_dir, exist_ok=True)
    os.makedirs(val_labels_dir, exist_ok=True)

    # Copy original training data and augment it
    train_img_dir = os.path.join(main_path, 'train', 'images')
    train_label_dir = os.path.join(main_path, 'train', 'labels')

    img_files = sorted(glob.glob(os.path.join(train_img_dir, '*.*')))
    print(f"Found {len(img_files)} original training images")

    # Define transformations for data augmentation
    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.RandomScale(scale_limit=0.2, p=0.4),
        A.Blur(blur_limit=3, p=0.2),
        A.MotionBlur(blur_limit=3, p=0.2),  # Helps with fast-moving balls
    ])

    # Copy and augment training files
    for img_path in img_files:
        img_name = os.path.basename(img_path)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(train_label_dir, label_name)

        # Copy original file
        shutil.copy(img_path, os.path.join(train_images_dir, img_name))
        shutil.copy(label_path, os.path.join(train_labels_dir, label_name))

        # Read label content to check if file contains rare objects
        with open(label_path, 'r') as f:
            labels = f.readlines()

        has_rare_class = False
        for label in labels:
            class_id = int(label.split()[0])
            if class_id in [0, 1, 3]:  # ball, goalkeeper, referee
                has_rare_class = True
                break

        # If rare objects are present, perform data augmentation
        if has_rare_class:
            img = np.array(PILImage.open(img_path))

            # Create 3 augmented versions for each image with rare object(s)
            for i in range(3):
                augmented = transform(image=img)
                aug_img = augmented['image']

                # Save augmented image
                aug_img_name = f"aug_{i}_{img_name}"
                PILImage.fromarray(aug_img).save(os.path.join(train_images_dir, aug_img_name))

                # Copy label (assuming augmentation does not change object positions)
                aug_label_name = f"aug_{i}_{label_name}"
                shutil.copy(label_path, os.path.join(train_labels_dir, aug_label_name))

    # Copy original validation data
    val_img_dir = os.path.join(main_path, 'valid', 'images')
    val_label_dir = os.path.join(main_path, 'valid', 'labels')

    val_files = sorted(glob.glob(os.path.join(val_img_dir, '*.*')))
    print(f"Found {len(val_files)} original validation images")

    # Copy all validation files
    for img_path in val_files:
        img_name = os.path.basename(img_path)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(val_label_dir, label_name)

        # Copy files
        shutil.copy(img_path, os.path.join(val_images_dir, img_name))
        if os.path.exists(label_path):  # Check if label file exists
            shutil.copy(label_path, os.path.join(val_labels_dir, label_name))

    # Update data.yaml with new paths
    aug_data_yaml = os.path.join(aug_dir, 'data.yaml')
    with open(custom_data_yaml, 'r') as infile, open(aug_data_yaml, 'w') as outfile:
        data = yaml.safe_load(infile)
        data['train'] = train_images_dir      # Update train path
        data['val'] = val_images_dir          # Update validation path
        data['path'] = aug_dir                # Update base path
        data['names'] = class_names           # Ensure class names are correct
        data['nc'] = len(class_names)         # Number of classes
        yaml.dump(data, outfile)

    return aug_data_yaml


In [24]:
augmented_data_yaml = create_augmented_dataset()


Found 250 original training images
Found 43 original validation images


In [25]:
print("\nCheck YAML content:")
with open(augmented_data_yaml, 'r') as f:
    print(f.read())



Check YAML content:
class_weights:
- 5.0
- 3.0
- 1.0
- 4.0
names:
- ball
- goalkeeper
- player
- referee
nc: 4
path: /content/football_detection_improved/augmented_dataset
roboflow:
  license: CC BY 4.0
  project: football-players-detection-3zvbc
  url: https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/9
  version: 9
  workspace: roboflow-jvuqo
test: ../test/images
train: /content/football_detection_improved/augmented_dataset/train/images
val: /content/football_detection_improved/augmented_dataset/valid/images



In [26]:
model = YOLO(model_variant)
model.to(DEVICE)


100%|██████████| 49.7M/49.7M [00:00<00:00, 285MB/s]


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_

In [27]:
results_train = model.train(
    data=augmented_data_yaml,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    name='yolov8_football_detection_improved',
    weight_decay=WEIGHT_DECAY,
    lr0=LEARNING_RATE,
    lrf=0.01,          # Use higher learning rate final value for cosine decay
    mosaic=1.0,        # Enable mosaic augmentation
    mixup=0.2,         # Add mixup augmentation
    copy_paste=0.3,    # Add copy-paste augmentation
    fliplr=0.5,        # Horizontal flip to create variations
    scale=0.25,        # Random scaling to improve detection at different object sizes
    degrees=5.0,       # Slight rotation (suitable for sports)
    exist_ok=True,
    patience=15,       # Use early stopping to avoid overfitting
    save_period=5      # Save checkpoint every 5 epochs
)

print("\n--- Training Completed ---")
print(f"Training results saved at: {results_train.save_dir}")


Ultralytics 8.3.163 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football_detection_improved/augmented_dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_football_detection_improved, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

100%|██████████| 755k/755k [00:00<00:00, 21.1MB/s]

Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3778012  ultralytics.nn.modules.head.Detect           [4, [192, 384, 576]]          
Model summary: 169 layers, 25,858,636 parameters, 25,858,620 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 86.5MB/s]


...

      46/50      12.5G     0.8486      1.012     0.8874        197       1280: 100%|██████████| 125/125 [02:07<00:00,  1.02s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.32it/s]

                   all         43       1025      0.931      0.816       0.88      0.615
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 31, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



46 epochs completed in 1.781 hours.
Optimizer stripped from runs/detect/yolov8_football_detection_improved/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/yolov8_football_detection_improved/weights/best.pt, 52.1MB

Validating runs/detect/yolov8_football_detection_improved/weights/best.pt...
Ultralytics 8.3.163 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.50it/s]


                   all         43       1025      0.888      0.854      0.876       0.64
                  ball         39         39      0.847      0.513      0.617      0.332
            goalkeeper         32         32      0.857      0.939      0.926      0.759
                player         43        853      0.971      0.982      0.987      0.795
               referee         43        101      0.875       0.98      0.975      0.673
Speed: 0.5ms preprocess, 24.3ms inference, 0.0ms loss, 6.3ms postprocess per image
Results saved to runs/detect/yolov8_football_detection_improved

--- Training Completed ---
Training results saved at: runs/detect/yolov8_football_detection_improved


In [ ]:
best_model_path = os.path.join(results_train.save_dir, 'weights/best.pt')
last_model_path = os.path.join(results_train.save_dir, 'weights/last.pt')

models = []
if os.path.exists(best_model_path):
    print(f"Loading best model from: {best_model_path}")
    best_model = YOLO(best_model_path)
    best_model.to(DEVICE)
    models.append(best_model)

if os.path.exists(last_model_path):
    print(f"Loading last model from: {last_model_path}")
    last_model = YOLO(last_model_path)
    last_model.to(DEVICE)
    models.append(last_model)

Loading best model from: runs/detect/yolov8_football_detection_improved/weights/best.pt
Loading last model from: runs/detect/yolov8_football_detection_improved/weights/last.pt
